## Magnetostatic problem


In [2]:
from ngsolve import *
from netgen.read_gmsh import ReadGmsh
from ngsolve.webgui import Draw
from netgen.csg import *
import math
import pyvista as pv
import numpy as np
from ngsolve.krylovspace import GMRes

mesh_path = '../../meshes/coil_box_named'
output_path = '../../output/case2/case2_ngsolve'
output_path_gauss = '../../output/case2/gauss/case2_ngsolve'

# Import geometries
mesh = ReadGmsh(mesh_path + ".msh")

# for i in range(1, 3):
#     # print(i)
#     mesh.SetMaterial(i, f'{i}')

# for i in range(1, 13):
#     # print(i)
#     mesh.SetBCName(i-1, f'{i}')

mesh = Mesh(mesh)

mesh.ngmesh.Save(mesh_path + ".vol")

In [3]:
mesh.ne, mesh.nv, mesh.GetMaterials(), mesh.GetBoundaries()

(69629, 12090, ('vacuum', 'wire'), ('CoilIn', 'CoilOut'))

In [5]:
# Define material, coil, and BC parameters

I_coil = 2191.9 # Input electric current [A]
sigma_coil = 62.83185 # Coil's electric conductivity [S/m]
mu_r_coil = 1.0 # Coil's relative permeability [-]
# mu0 = 4*math.pi*1e-7

mu0 = 1.0

print(1/mu0)

print(1/(4*math.pi))

print(1/12.5663706212)

1.0
0.07957747154594767
0.07957747150262762


In [64]:
sigma = {"vacuum": 0.0, "wire": sigma_coil}  # Electric conductivity [S/m]
mu_r = {"vacuum": 1.0, "wire": mu_r_coil} # Relative permeability [-]

sigma_cf = CoefficientFunction([sigma.get(mat, 0.0) for mat in mesh.GetMaterials()])
mu_cf = mu0 * CoefficientFunction([mu_r.get(mat, 1.0) for mat in mesh.GetMaterials()])

crosssection = Integrate(1, mesh, definedon=mesh.Boundaries("CoilIn"))

print(f"Coil cross section = {crosssection} m^2.")

r = np.sqrt(crosssection / math.pi)

print(f"Coil radius = {r} m.")

A_nom = mu0 * I_coil / (4 * math.pi * r)

print(f"Scale of A = {A_nom} Wb/m.")

fespot = H1(mesh, order=1, definedon=mesh.Materials("wire"), dirichlet="CoilOut")
phi,psi = fespot.TnT()
with TaskManager():
    bfa = BilinearForm(sigma_cf*grad(phi)*grad(psi)*dx).Assemble()
    inv = bfa.mat.Inverse(freedofs=fespot.FreeDofs(), inverse="sparsecholesky")
    lff = LinearForm(I_coil/crosssection*psi*ds("CoilIn")).Assemble()
    gfphi = GridFunction(fespot)
    gfphi.vec.data = inv * lff.vec

gfcurrdens = -sigma_cf*grad(gfphi)

Coil cross section = 7.0710678118656e-05 m^2.
Coil radius = 0.004744249983287986 m.
Scale of A = 36765.73968403695 Wb/m.


In [65]:
fespot_global = H1(mesh, order=1)
gfphi_global = GridFunction(fespot_global)
gfphi_global.Set(gfphi, definedon=mesh.Materials("wire"))

# fescurrden_global = VectorH1(mesh, order=1)
fescurrden_global = VectorL2(mesh, order=0)
# fescurrden_global = HCurl(mesh, order=1)
# fescurrden_global = HDiv(mesh, order=1)
gfcurrdens_global = GridFunction(fescurrden_global)
gfcurrdens_global.Set(gfcurrdens, definedon=mesh.Materials("wire"))

In [66]:
# fes = HCurl(mesh, order=1, complex=True, dirichlet="VacuumSurface", nograds = False)

# fes = HCurl(mesh, order=1, nograds=True)

# fes = HCurl(mesh, order=1, gradientdomains="wire")

# fes = HCurl(mesh, order=1, nograds=True)

# fes = HCurl(mesh, order=1, dirichlet="Side1|Side2|Side3|Side4|Side5|Side6", nograds=True)


fes = HCurl(mesh, order=1, dirichlet="CoilIn|CoilOut|1|2|3|4|5|12", nograds=False) # THIS ONE

# fes = HCurl(mesh, order=1, dirichlet="1|2|3|4|5|12", nograds=False)

print ("HCurl dofs:", fes.ndof) 
u,v = fes.TnT()
a = BilinearForm(1/mu_cf*curl(u)*curl(v)*dx+1e-7/mu_cf*u*v*dx)

# pre = preconditioners.BDDC(a)
# pre = preconditioners.HCurlAMG(a)
pre = preconditioners.MultiGrid(a)
# pre = BilinearForm(u*v*ds, diagonal=True).Assemble().mat.Inverse()
# pre = preconditioners.Local(a)
# f = LinearForm(sigma_cf*grad(gfphi)*v*dx("wire"))
f = LinearForm(-gfcurrdens_global*v*dx("wire"))
with TaskManager():
    a.Assemble()
    f.Assemble()

HCurl dofs: 165218
hcurl smoothingblocks, SmoothingType = 2


In [67]:
A = GridFunction(fes)
A.vec[:] = 0
# pre = preconditioners.Local(a)
A.vec.data = GMRes(a.mat, f.vec, pre=pre, maxsteps=200, printrates=True, tol= 1e-9)

# A.vec.data = GMRes(a.mat, f.vec, freedofs=fes.FreeDofs(), maxsteps=100, printrates=True, tol= 1e-9)

# inv = CGSolver(a.mat, pre, precision=1e-9, maxsteps=100, printrates=True)
# A.vec.data = inv * f.vec

# A.vec.data = solvers.CG(a.mat, f.vec, pre=pre, tol= 1e-9)

# solvers.BVP(bf=a, lf=f, gf=A, pre=pre, \
#             solver=solvers.CGSolver, solver_flags={"plotrates": True, "tol" : 1e-12})

# solvers.BVP(bf=a, lf=f, gf=A, pre=pre, \
#             solver=solvers.CGSolver, solver_flags={"plotrates": True, "tol" : 1e-12})

GMRes iteration 1, residual = 10216881584666.453     
GMRes iteration 2, residual = 2161550809778.0005     
GMRes iteration 3, residual = 127023274664.71103     
GMRes iteration 4, residual = 10354628690.740767     
GMRes iteration 5, residual = 1409920671.5781028     
GMRes iteration 6, residual = 102598176.39059262     
GMRes iteration 7, residual = 5770787.039058586     
GMRes iteration 8, residual = 331947.84835644317     
GMRes iteration 9, residual = 15692.236153806976     
GMRes iteration 10, residual = 812.1987319818647     
GMRes iteration 11, residual = 59.147589315192306     
GMRes iteration 12, residual = 4.329310798044933     
GMRes iteration 13, residual = 0.2744177741935526     
GMRes iteration 14, residual = 0.022608246229946144     
GMRes iteration 15, residual = 0.016321391834959918     
GMRes iteration 16, residual = 0.016303139272818432     
GMRes iteration 17, residual = 0.01630307962385506     
GMRes iteration 18, residual = 0.016303079335312214     
GMRes iterati

In [68]:
fes_A = VectorH1(mesh, order=3)
A_gf = GridFunction(fes_A)
A_gf.Set(A.real)

In [69]:
B = curl(A)
# fes_B = VectorH1(mesh, order=3)
# B_gf = GridFunction(fes_B)
# B_gf.Set(B.real)

fes_B = HDiv(mesh, order=1)
B_gf = GridFunction(fes_B)
B_gf.Set(B.real)

In [70]:
# fespot_global = H1(mesh, order=1)
# gfphi_global = GridFunction(fespot_global)
# gfphi_global.Set(gfphi, definedon=mesh.Materials("wire"))

# fescurrden_global = VectorH1(mesh, order=1)
# # fescurrden_global = VectorL2(mesh, order=0)
# # fescurrden_global = HCurl(mesh, order=1)
# # fescurrden_global = HDiv(mesh, order=1)
# gfcurrdens_global = GridFunction(fescurrden_global)
# gfcurrdens_global.Set(gfcurrdens, definedon=mesh.Materials("wire"))

In [71]:
# vtk = VTKOutput(mesh,coefs=[gfphi],names=["sol"],filename=output_path + "electric_potential",subdivision=0)
# vtk.Do()

# res = pv.read(mesh_path + ".msh")
# points = res.points
# gfphi_out = np.zeros((points.shape[0], 1))
# gfcurrdens_out = np.zeros((points.shape[0], 3))
# B_sol = np.zeros((points.shape[0], 3))
# A_sol = np.zeros((points.shape[0], 3))

# for i in range(points.shape[0]):
#     point = mesh(points[i, 0], points[i, 1], points[i, 2])
#     gfphi_out[i, :] = gfphi_global(point)
#     B_sol[i, :] = B_gf(point)
#     A_sol[i, :] = A_gf(point)
#     gfcurrdens_out[i, :] = gfcurrdens_global(point)

# res["electric_potential"] = gfphi_out
# res["magnetic_flux_density"] = B_sol
# res["magnetic_vector_potential"] = A_sol
# res["current_density"] = gfcurrdens_out

# res.save(output_path + ".vtu")



vtk = VTKOutput(mesh,coefs=[A, B, gfcurrdens, A_gf, gfphi, B_gf],
                names=["magnetic_vector_potential_nd", "magnetic_flux_density_nd", "current_density", "magnetic_vector_potential", "electric_potential", "magnetic_flux_density"],
                filename=output_path, subdivision=0)
vtk.Do()

'../../output/case2/case2_ngsolve'

In [73]:

from pprint import pprint
order_ir = 1 # Integration order
points_per_elem = 1 # Number of Gauss points per element
ir = IntegrationRule(ET.TET, order=order_ir)

gauss_coords = np.zeros((mesh.ne*points_per_elem, 3))
gauss_values1 = np.zeros((mesh.ne*points_per_elem, 3))
gauss_values2 = np.zeros((mesh.ne*points_per_elem, 3))
gauss_values3 = np.zeros((mesh.ne*points_per_elem, 1))

for i, el in enumerate(mesh.Elements(VOL)):
    # Get the transformation for the element
    trafo = mesh.GetTrafo(el)
    # Coordinates
    # coords = trafo(ir)
    # print(coords[0])
    # Values corresponding to the coordinates
    # vals = A(trafo(ir))


    for j, ip in enumerate(ir):
        # Coordinate
        coord = trafo(ip)
        # Values corresponding to the coordinate
        val1 = A(trafo(ip))
        val2 = B(trafo(ip))
        val3 = gfphi(trafo(ip))

        gauss_coords[i+j, :] = coord.point
        gauss_values1[i+j, :] = val1
        gauss_values2[i+j, :] = val2
        gauss_values3[i+j] = val3

    # if i == 60:
    #    pprint(dir(coord))
    #    print(coord.point)

results = np.concatenate((gauss_coords, gauss_values1, gauss_values2, gauss_values3), axis=1)

print(results.shape)

print("Min and max X coord:")
print(np.min(gauss_coords[:, 0]))
print(np.max(gauss_coords[:, 0]))

print("Min and max Y coord:")
print(np.min(gauss_coords[:, 1]))
print(np.max(gauss_coords[:, 1]))

print("Min and max Z coord:")
print(np.min(gauss_coords[:, 2]))
print(np.max(gauss_coords[:, 2]))

np.save(output_path_gauss + ".npy", results)

# np.save("coords.npy", gauss_coords)
# np.save("vals.npy", gauss_values)

(69629, 10)
Min and max X coord:
-0.09800790594122501
0.0979479051960425
Min and max Y coord:
-0.099551605255125
0.09795298646876001
Min and max Z coord:
-0.09789693344965
0.09789017226766
